# TP - Qualité des données : valeurs extrêmes
## Spaceship Titanic - notebook corrigé

**Durée indicative : 1 h 30**

### Objectifs

À partir d'un jeu de données biométriques volontairement détérioré, l'objectif est de :

1. explorer une distribution et repérer visuellement des observations atypiques ;
2. commencer par des **règles métier** avant d'utiliser des méthodes statistiques ;
3. appliquer et comparer **IQR** et **z-score** ;
4. comprendre qu'une valeur atypique n'est pas nécessairement une erreur ;
5. montrer qu'une anomalie peut dépendre de la **population** étudiée ;
6. détecter des anomalies **multivariées** à partir de la taille et du poids ;
7. si le temps le permet, tester une méthode avancée avec **Isolation Forest** et éventuellement **LOF** ;
8. distinguer la détection d'une anomalie de la décision de la corriger, la conserver ou l'exclure.

## 0. Contexte

Avant le départ du *Spaceship Titanic*, chaque passager passe un contrôle biométrique obligatoire afin d'adapter certains équipements de sécurité et d'évacuation.

Un fichier spécifique contient :

- `PassengerId` : identifiant du passager ;
- `gender` : genre ;
- `height_cm` : taille en centimètres ;
- `weight_kg` : poids en kilogrammes.

Le fichier a été volontairement détérioré : certaines mesures sont erronées, certaines sont seulement rares, et d'autres ne deviennent suspectes qu'en combinant plusieurs variables.

Votre mission consiste à **détecter les observations à investiguer, puis à décider ce qu'elles signifient**.

> Une méthode statistique ne sait pas si une valeur est vraie ou fausse : elle signale seulement qu'elle est inhabituelle.

## 1. Chargement et première exploration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

DATA_PATH = Path("../data/passenger_biometrics.csv")
SPACESHIP_PATH = Path("../../01_missing/data/spaceship_titanic_missing.csv")
GROUND_TRUTH_PATH = Path("../data/passenger_biometrics_ground_truth.csv")

df = pd.read_csv(DATA_PATH)
spaceship = pd.read_csv(SPACESHIP_PATH)

print("Dimensions biométrie :", df.shape)
display(df.head())

### Exercice 1 — Comprendre le dataset

1. Affichez les types des variables.
2. Vérifiez l'unicité de `PassengerId`.
3. Vérifiez la présence éventuelle de valeurs manquantes.
4. Affichez quelques statistiques descriptives sur `height_cm` et `weight_kg`.
5. Affichez la répartition de `gender`.

**Question :** voyez-vous déjà des valeurs qui semblent suspectes ?

In [ ]:
# Votre réponse

## 2. Relier les mesures aux passagers

Le fichier biométrique contient `PassengerId`, qui est également présent dans le dataset principal du *Spaceship Titanic*.

### Exercice 2

1. Vérifiez que tous les `PassengerId` du fichier biométrique existent dans le dataset principal.
2. Réalisez une jointure afin d'ajouter au moins `Name` et `HomePlanet` aux données biométriques.
3. Affichez quelques lignes du résultat.

Le but est simplement de vérifier que les mesures biométriques peuvent être rattachées aux passagers.

In [ ]:
# Votre réponse

## 3. Commencer par les règles métier

Avant IQR, z-score ou machine learning, certaines anomalies peuvent être détectées avec des règles simples.

Pour ce TP, on retient volontairement des bornes **larges** :

- une taille inférieure à 100 cm ou supérieure à 230 cm mérite une investigation ;
- un poids inférieur ou égal à 0 kg ou supérieur à 250 kg mérite une investigation.

Ces seuils ne définissent pas un humain « normal ». Ils servent seulement à détecter des valeurs manifestement incohérentes dans le contexte du contrôle biométrique.

### Exercice 3

1. Construisez un masque booléen `business_rule_flag` à partir de ces règles.
2. Comptez le nombre d'observations signalées.
3. Affichez les lignes concernées.
4. Pour chaque ligne, essayez de formuler une hypothèse : erreur de saisie, unité, inversion, etc.

In [ ]:
# Votre réponse

## 4. Visualiser les distributions

### Exercice 4

Pour `height_cm` et `weight_kg` :

1. tracez un histogramme ;
2. tracez un boxplot ;
3. repérez visuellement les observations extrêmes.

**Question :** les deux graphiques donnent-ils la même information ?

In [ ]:
# Votre réponse

## 5. Détection avec l'IQR

Pour une variable $x$ :

- $IQR = Q3 - Q1$ ;
- borne basse : $Q1 - 1.5 \times IQR$ ;
- borne haute : $Q3 + 1.5 \times IQR$.

### Exercice 5

1. Écrivez une fonction `iqr_mask()` qui retourne :
   - un masque booléen ;
   - la borne basse ;
   - la borne haute.
2. Appliquez-la à `height_cm` puis à `weight_kg`.
3. Comptez le nombre de candidats détectés pour chaque variable.
4. Affichez les observations signalées par au moins l'une des deux variables.

In [ ]:
# Votre réponse

### Exercice 6 — Comparer règles métier et IQR

Comparez les identifiants détectés par les deux approches :

- uniquement par les règles métier ;
- uniquement par l'IQR ;
- par les deux méthodes.

**Question :** pourquoi l'IQR peut-il signaler des valeurs que les règles métier considèrent pourtant comme possibles ?

In [ ]:
# Votre réponse

## 6. Détection avec le z-score

Le z-score mesure la distance à la moyenne en nombre d'écarts-types.

### Exercice 7

1. Calculez un z-score pour `height_cm` et `weight_kg`.
2. Signalez les observations telles que $|z| > 3$.
3. Comparez le nombre de candidats avec l'IQR.
4. Identifiez les observations sur lesquelles IQR et z-score ne sont pas d'accord.

In [ ]:
# Votre réponse

## 7. Une anomalie dépend de la population

Le dataset contient deux populations dont les distributions de taille et de poids ne sont pas identiques.

### Exercice 8

1. Tracez un boxplot de `height_cm` en séparant `gender`.
2. Faites de même pour `weight_kg`.
3. Calculez ensuite les bornes IQR **séparément pour chaque population**.
4. Comparez avec l'IQR calculé sur l'ensemble du dataset.

**Question :** pourquoi une segmentation pertinente peut-elle modifier la détection des valeurs atypiques ?

In [ ]:
# Votre réponse

## 8. Une anomalie peut être multivariée

Une taille de 195 cm est possible. Un poids de 48 kg est également possible. Pourtant, leur combinaison peut être particulièrement inhabituelle.

### Exercice 9

1. Tracez un nuage de points `height_cm` × `weight_kg`.
2. Colorez les observations par `gender`.
3. Repérez les points isolés ou les associations taille / poids étonnantes.
4. Comparez ces points avec les résultats de l'IQR et du z-score.

**Question :** certaines observations semblent-elles anormales uniquement lorsque les deux variables sont considérées ensemble ?

In [ ]:
# Votre réponse

## 9. Méthode avancée : Isolation Forest

Isolation Forest cherche les observations qui sont faciles à isoler du reste du jeu de données.

On utilise ici uniquement :

- `height_cm` ;
- `weight_kg`.

### Exercice 10

1. Entraînez un `IsolationForest` avec une contamination de `0.01`.
2. Ajoutez au dataset :
   - le label prédit ;
   - le score d'anomalie.
3. Affichez les 20 observations les plus atypiques.
4. Représentez les candidats sur le nuage de points.

**Question :** la méthode retrouve-t-elle uniquement des erreurs manifestes ?

In [ ]:
# Votre réponse

## 10. Bonus — Local Outlier Factor

Cette partie est optionnelle si le temps le permet.

LOF compare la densité locale d'une observation à celle de ses voisins. Il peut donc repérer des anomalies qui sont atypiques **localement**.

### Exercice 11 — Bonus

1. Standardisez `height_cm` et `weight_kg`.
2. Appliquez `LocalOutlierFactor` avec `n_neighbors=35` et `contamination=0.01`.
3. Affichez les observations ayant les scores les plus élevés.
4. Comparez rapidement avec Isolation Forest.

In [ ]:
# Votre réponse

### Bonus — DBSCAN : détecter les points hors des zones denses

DBSCAN (*Density-Based Spatial Clustering of Applications with Noise*) n'est pas, à l'origine, un algorithme de détection d'outliers.

Il cherche plutôt à identifier des **groupes de points suffisamment denses**. Les observations qui ne peuvent être rattachées à aucune zone dense sont classées comme du **bruit** (`-1`).

Dans notre cas, cela apporte une lecture différente :

- Isolation Forest cherche les observations faciles à isoler ;
- LOF compare la densité locale d'un point avec celle de ses voisins ;
- DBSCAN cherche des zones denses et considère le reste comme du bruit.

Avant d'utiliser DBSCAN, il faut cependant traiter un point essentiel : **l'échelle des variables**.

#### Pourquoi standardiser ?

La distance entre les points est au cœur de DBSCAN.

Or ici :

- `height_cm` est exprimé en centimètres ;
- `weight_kg` est exprimé en kilogrammes.

Même si les ordres de grandeur sont proches, les deux variables n'ont ni la même unité ni exactement la même dispersion.

On standardise donc les variables avant d'appliquer DBSCAN :

```text
valeur standardisée = (valeur - moyenne) / écart-type
```

Après standardisation, les deux variables sont ramenées sur une échelle comparable :

- moyenne proche de `0` ;
- écart-type proche de `1`.

Cela évite qu'une variable influence artificiellement les distances davantage que l'autre.

In [ ]:
# Votre réponse

#### Appliquer DBSCAN

Deux paramètres contrôlent principalement son comportement :

- `eps` : rayon maximal autour d'un point pour considérer d'autres observations comme voisines ;
- `min_samples` : nombre minimal de points nécessaires pour former une zone dense.

Il n'existe pas de valeur universelle : ces paramètres dépendent de la densité et de l'échelle des données.

Pour ce TP, on utilise des valeurs simples afin d'observer le principe sans chercher à optimiser finement l'algorithme.

In [ ]:
# Votre réponse

#### Comparer Isolation Forest et DBSCAN

Les deux méthodes sélectionnent-elles les mêmes observations ?

In [ ]:
# Votre réponse

## 11. Détecter n'est pas décider

Vous disposez maintenant de plusieurs détecteurs :

- règles métier ;
- IQR ;
- z-score ;
- analyse graphique bivariée ;
- Isolation Forest ;
- éventuellement LOF.

### Exercice 12 — Décider quoi faire

Choisissez au moins **5 observations** parmi les candidats détectés et construisez un tableau contenant :

- `PassengerId` ;
- `height_cm` ;
- `weight_kg` ;
- méthode(s) ayant signalé l'observation ;
- votre interprétation ;
- votre décision parmi :
  - **conserver** ;
  - **corriger** ;
  - **exclure** ;
  - **investiguer davantage**.

Justifiez brièvement chaque décision.

In [ ]:
# Votre réponse

## 12. Nettoyer les données

La détection n'est qu'une étape. Nous allons maintenant construire une version **nettoyée** du dataset.

L'objectif n'est pas de supprimer tous les outliers :

- certaines erreurs peuvent être corrigées lorsque leur origine est suffisamment claire ;
- certaines observations doivent seulement être marquées pour investigation ;
- les valeurs rares mais plausibles doivent rester dans le dataset.

Nous allons travailler dans une nouvelle variable `df_clean` afin de conserver `df` intact.

In [ ]:
# Votre réponse

### Exercice 13 — Corriger une inversion probable taille / poids

Recherchez les lignes où :

- la taille est anormalement faible pour un adulte ;
- le poids est très élevé ;
- les deux valeurs semblent plausibles si elles sont inversées.

Corrigez uniquement ce cas.

In [ ]:
# Votre réponse

### Exercice 14 — Corriger les erreurs d'unité ou de décimale

Traitez séparément les cas suivants lorsqu'ils sont suffisamment évidents :

1. une taille exprimée en **mètres** alors que la colonne attend des centimètres ;
2. une taille manifestement affectée par un décalage de décimale ;
3. un poids manifestement affecté par un décalage de décimale.

Avant chaque correction, affichez les lignes concernées.

In [ ]:
# Votre réponse

### Exercice 15 — Corriger une valeur impossible

Un poids négatif est techniquement impossible.

Dans ce dataset pédagogique, l'erreur de signe est suffisamment évidente pour proposer une correction.  
Dans un cas réel, si l'origine de l'erreur n'était pas certaine, on préférerait généralement remplacer la valeur par `NaN` et investiguer.

In [ ]:
# Votre réponse

### Exercice 16 — Ne pas supprimer les valeurs seulement parce qu'elles sont atypiques

Après les corrections précédentes :

1. vérifiez à nouveau les règles métier ;
2. identifiez les observations encore inhabituelles ;
3. **ne les supprimez pas automatiquement**.

Ajoutez plutôt une colonne booléenne `needs_review` pour signaler les observations qui mériteraient encore une investigation.

Vous pouvez notamment vous appuyer sur Isolation Forest.

In [ ]:
# Votre réponse

### Bonus — Rejouer DBSCAN après nettoyage

Comme pour Isolation Forest, il est utile de relancer DBSCAN après correction des erreurs manifestes.

La standardisation doit être recalculée à partir de `df_clean`, car la distribution des variables a changé après nettoyage.

In [ ]:
# Votre réponse

## 13. Rejouer l'analyse après nettoyage

Les valeurs manifestement erronées écrasaient les échelles des premiers graphiques.

Reproduisez maintenant :

- les histogrammes de taille et de poids ;
- les boxplots par `gender` ;
- le nuage de points `height_cm × weight_kg`.

Comparez visuellement les résultats avec les graphiques obtenus avant nettoyage.

In [ ]:
# Votre réponse

## 15. Synthèse

À retenir :

1. **commencer par les règles métier** ;
2. visualiser la distribution avant d'appliquer une méthode automatique ;
3. IQR et z-score ne reposent pas sur les mêmes hypothèses ;
4. un outlier n'est pas nécessairement une erreur ;
5. la population de référence modifie ce qui paraît atypique ;
6. certaines anomalies sont uniquement visibles en combinant plusieurs variables ;
7. Isolation Forest, LOF et DBSCAN proposent des visions différentes de l'anomalie ;
8. les méthodes avancées produisent des **candidats**, pas des décisions ;
9. le traitement final dépend du contexte : conserver, corriger, exclure ou investiguer ;
10. **corriger les erreurs identifiables avant de rejouer les analyses** améliore fortement la lisibilité sans supprimer arbitrairement les cas rares.